# Bank Statement Transaction Extraction with Llama Vision

Specialized notebook for extracting transactional data from bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Handles empty debit/credit cells correctly
- Preserves row-by-row transaction structure
- Calculates running balances
- Validates transaction integrity

In [1]:
# Configuration
from pathlib import Path
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Dynamically discover all PNG images in evaluation_data directory
EVALUATION_DATA_DIR = Path("evaluation_data")

# Automatically find all PNG files in the evaluation_data directory
BANK_STATEMENT_IMAGES = {}
if EVALUATION_DATA_DIR.exists():
    for png_file in sorted(EVALUATION_DATA_DIR.glob("*.png")):
        # Use the filename (without extension) as the key
        key = png_file.stem.upper()
        BANK_STATEMENT_IMAGES[key] = str(png_file)

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# Model loading settings - ENABLE 8-BIT QUANTIZATION FOR V100 PRODUCTION
USE_QUANTIZATION = True   # ENABLED for V100 production deployment
LOAD_IN_8BIT = True       # Required for V100 memory constraints
DEVICE_MAP = "auto"       # Automatic device mapping
MAX_NEW_TOKENS = 4000     # REDUCED for V100 - was 6000, now 4000 to prevent OOM

# Rich imports for pretty printing
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.progress import track
from rich import print as rprint
from rich.syntax import Syntax

# Initialize Rich console
console = Console()
rprint("[bold blue]🚀 Configuration loaded with Rich formatting enabled[/bold blue]")
rprint("[yellow]⚠️ V100 Production Mode: 8-bit quantization ENABLED[/yellow]")

# Display discovered images in a nice table
if BANK_STATEMENT_IMAGES:
    rprint(f"[green]✅ Found {len(BANK_STATEMENT_IMAGES)} bank statement images[/green]")
    
    image_table = Table(title="📄 Discovered Bank Statement Images", border_style="cyan")
    image_table.add_column("Key", style="yellow", no_wrap=True)
    image_table.add_column("File Path", style="green")
    
    for key, path in BANK_STATEMENT_IMAGES.items():
        image_table.add_row(key, path)
    
    console.print(image_table)
else:
    rprint("[yellow]⚠️ No PNG images found in evaluation_data directory[/yellow]")
    rprint("[yellow]Please ensure PNG files exist in the evaluation_data/ directory[/yellow]")

🚀 Configuration loaded with Rich formatting enabled

⚠️ V100 Production Mode: 8-bit quantization ENABLED

✅ Found 11 bank statement images

                  📄 Discovered Bank Statement Images                  
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Key                    ┃ File Path                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ ANZ_STATEMENT_001      │ evaluation_data/anz_statement_001.png      │
│ COMMBANK_STATEMENT_001 │ evaluation_data/commbank_statement_001.png │
│ IMAGE_001              │ evaluation_data/image_001.png              │
│ IMAGE_002              │ evaluation_data/image_002.png              │
│ IMAGE_003              │ evaluation_data/image_003.png              │
│ IMAGE_004              │ evaluation_data/image_004.png              │
│ IMAGE_005              │ evaluation_data/image_005.png              │
│ IMAGE_006              │ evaluation_data/image_006.png              │
│ IMAGE_007              │ evaluation_data/image_007.png              │
│ NAB_STATEMENT_001      │ evaluation_data/nab_statement_001.png      │
│ WESTPAC_STATEMENT_001  │ evaluation_data/westpac_statement_001.png  │
└────────────────────────┴────────────────────────────────────────────┘

# Configuration & Setup

This section handles the initial setup including dynamic image discovery, model configuration, and Rich console initialization.

# Core Library Imports

Essential imports for Vision Language Model processing, image handling, and data manipulation.

In [2]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from PIL import Image
import json
import re
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from datetime import datetime
from decimal import Decimal, InvalidOperation
import warnings

warnings.filterwarnings('ignore')

rprint("[bold green]✅ Core libraries imported successfully[/bold green]")

✅ Core libraries imported successfully

# Image Selection & Validation

Select and validate the bank statement image for processing, with fallback to available images.

In [3]:
# =============================================================================
# IMAGE SELECTION & VALIDATION
# =============================================================================

# Intelligent image selection for testing - prioritizes bank statements
rprint("[bold blue]🎯 Selecting test image for extraction...[/bold blue]")

def select_test_image() -> str:
    """
    Intelligently select the best test image, prioritizing actual bank statements.
    
    Returns:
        Path to selected test image
    """
    if not BANK_STATEMENT_IMAGES:
        rprint("[red]❌ No images found in evaluation_data directory[/red]")
        return None
    
    # Priority order for test image selection
    priority_patterns = [
        # Real bank statements (highest priority)
        'COMMBANK_STATEMENT',
        'ANZ_STATEMENT', 
        'NAB_STATEMENT',
        'WESTPAC_STATEMENT',
        # Generic statements
        'STATEMENT',
        # Fallback to any image
        'IMAGE_'
    ]
    
    # Try to find image matching priority patterns
    for pattern in priority_patterns:
        matching_keys = [key for key in BANK_STATEMENT_IMAGES.keys() if pattern in key]
        if matching_keys:
            selected_key = matching_keys[0]  # Take first match
            selected_path = BANK_STATEMENT_IMAGES[selected_key]
            
            rprint(f"[green]✅ Selected: {selected_key}[/green]")
            rprint(f"[cyan]📄 Path: {selected_path}[/cyan]")
            rprint(f"[blue]🎯 Reason: Matched priority pattern '{pattern}'[/blue]")
            
            return selected_path
    
    # Fallback - take first available image
    fallback_key = list(BANK_STATEMENT_IMAGES.keys())[0]
    fallback_path = BANK_STATEMENT_IMAGES[fallback_key]
    
    rprint(f"[yellow]⚠️ Using fallback: {fallback_key}[/yellow]")
    rprint(f"[cyan]📄 Path: {fallback_path}[/cyan]")
    
    return fallback_path

# Execute image selection
STATEMENT_IMAGE_PATH = select_test_image()

if STATEMENT_IMAGE_PATH:
    # Validate selected image exists
    image_path = Path(STATEMENT_IMAGE_PATH)
    if image_path.exists():
        rprint(f"[bold green]🎉 Test image ready: {image_path.name}[/bold green]")
        
        # Display image info
        try:
            from PIL import Image
            with Image.open(image_path) as img:
                width, height = img.size
                format_info = img.format or "Unknown"
                
                image_info_table = Table(title="📊 Selected Image Information", border_style="green")
                image_info_table.add_column("Property", style="cyan")
                image_info_table.add_column("Value", style="yellow")
                
                image_info_table.add_row("Filename", image_path.name)
                image_info_table.add_row("Format", format_info)
                image_info_table.add_row("Dimensions", f"{width} × {height} pixels")
                image_info_table.add_row("File Size", f"{image_path.stat().st_size / 1024:.1f} KB")
                image_info_table.add_row("Full Path", str(image_path))
                
                console.print(image_info_table)
                
        except Exception as e:
            rprint(f"[yellow]⚠️ Could not read image info: {e}[/yellow]")
    else:
        rprint(f"[red]❌ Selected image file does not exist: {image_path}[/red]")
        STATEMENT_IMAGE_PATH = None
else:
    rprint("[red]❌ No suitable test image found[/red]")

console.rule("[bold blue]Image Selection Complete[/bold blue]")

🎯 Selecting test image for extraction...

✅ Selected: COMMBANK_STATEMENT_001

📄 Path: evaluation_data/commbank_statement_001.png

🎯 Reason: Matched priority pattern 'COMMBANK_STATEMENT'

🎉 Test image ready: commbank_statement_001.png

               📊 Selected Image Information               
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property   ┃ Value                                      ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Filename   │ commbank_statement_001.png                 │
│ Format     │ PNG                                        │
│ Dimensions │ 800 × 1200 pixels                          │
│ File Size  │ 94.6 KB                                    │
│ Full Path  │ evaluation_data/commbank_statement_001.png │
└────────────┴────────────────────────────────────────────┘

──────────────────────────────────────────── Image Selection Complete ─────────────────────────────────────────────

# Model Loading & Validation

Load the Llama 3.2 Vision model with comprehensive validation and GPU optimization for V100.

In [4]:
rprint("[bold yellow]🚀 Loading Llama Vision model with V100 production optimizations...[/bold yellow]")

try:
    # Validate model path first
    if not Path(MODEL_PATH).exists():
        raise FileNotFoundError(f"Model path does not exist: {MODEL_PATH}")
    
    # Configure modern quantization for V100 compatibility
    if USE_QUANTIZATION and LOAD_IN_8BIT:
        rprint("[yellow]🔧 Configuring 8-bit quantization with BitsAndBytesConfig[/yellow]")
        
        from transformers import BitsAndBytesConfig
        
        # Modern quantization configuration
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0,  # Default threshold
            llm_int8_has_fp16_weight=False,  # Use int8 weights
            llm_int8_enable_fp32_cpu_offload=False  # Keep on GPU
        )
        
        rprint("[green]✅ Modern BitsAndBytesConfig configured[/green]")
    else:
        quantization_config = None
        rprint("[blue]ℹ️ No quantization - loading in full precision[/blue]")
    
    # Load model with unified configuration
    rprint("[dim]Loading Llama-3.2-Vision model...[/dim]")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
        quantization_config=quantization_config,  # Modern API
        low_cpu_mem_usage=True,  # Memory optimization
    )
    
    # Load processor
    rprint("[dim]Loading processor...[/dim]")
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    
    # Comprehensive model validation
    if model is None:
        raise RuntimeError("Model failed to load - returned None")
    if processor is None:
        raise RuntimeError("Processor failed to load - returned None")
    
    # Model parameter analysis
    model_params = sum(p.numel() for p in model.parameters())
    if model_params < 1e9:  # Less than 1B parameters seems wrong
        rprint(f"[yellow]⚠️ Warning: Model has unusually few parameters: {model_params:,.0f}[/yellow]")
    
    # Device placement validation
    model_device = next(model.parameters()).device
    if model_device.type == 'cpu' and torch.cuda.is_available():
        rprint("[yellow]⚠️ Warning: Model loaded on CPU despite CUDA availability[/yellow]")
    
    rprint("[bold green]✅ Model and processor loaded successfully![/bold green]")
    rprint(f"[cyan]📊 Device: {model_device}[/cyan]")
    
    # GPU memory analysis and V100-specific validation
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name()
        memory_allocated = torch.cuda.memory_allocated() / 1e9
        memory_reserved = torch.cuda.memory_reserved() / 1e9
        memory_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        rprint(f"[magenta]🎮 GPU: {gpu_name}[/magenta]")
        rprint(f"[blue]💾 Memory Allocated: {memory_allocated:.2f}GB[/blue]")
        rprint(f"[blue]💾 Memory Reserved: {memory_reserved:.2f}GB[/blue]")
        rprint(f"[blue]💾 Total GPU Memory: {memory_total:.0f}GB[/blue]")
        
        # V100-specific memory warnings (V100 has ~16GB)
        memory_usage_pct = memory_allocated / memory_total * 100
        if "V100" in gpu_name:
            if memory_usage_pct > 80:
                rprint(f"[red]🚨 CRITICAL V100 memory usage: {memory_usage_pct:.1f}% - REDUCE MAX_NEW_TOKENS[/red]")
            elif memory_usage_pct > 60:
                rprint(f"[yellow]⚠️ HIGH V100 memory usage: {memory_usage_pct:.1f}% - Monitor closely[/yellow]")
            elif memory_usage_pct > 40:
                rprint(f"[yellow]⚠️ Moderate V100 memory usage: {memory_usage_pct:.1f}%[/yellow]")
            else:
                rprint(f"[green]✅ Excellent V100 memory usage: {memory_usage_pct:.1f}%[/green]")
        else:
            # H200 or other high-memory GPU
            if memory_usage_pct > 70:
                rprint(f"[red]⚠️ High GPU memory usage: {memory_usage_pct:.1f}%[/red]")
            elif memory_usage_pct > 50:
                rprint(f"[yellow]⚠️ Moderate GPU memory usage: {memory_usage_pct:.1f}%[/yellow]")
            else:
                rprint(f"[green]✅ Good GPU memory usage: {memory_usage_pct:.1f}%[/green]")
    else:
        rprint("[yellow]⚠️ CUDA not available - using CPU (significantly slower)[/yellow]")
    
    # Comprehensive configuration validation table
    config_table = Table(title="🔧 Production Model Configuration", border_style="blue")
    config_table.add_column("Setting", style="cyan", no_wrap=True)
    config_table.add_column("Value", style="yellow")
    config_table.add_column("Status", style="green")
    
    # Configuration details
    config_table.add_row("Model Path", MODEL_PATH, "✅ Valid")
    config_table.add_row("Device Placement", str(model_device), "✅ Loaded")
    config_table.add_row("Quantization", "8-bit BitsAndBytesConfig" if USE_QUANTIZATION else "Disabled", 
                        "✅ Modern API" if USE_QUANTIZATION else "✅ Full Precision")
    config_table.add_row("Max New Tokens", str(MAX_NEW_TOKENS), 
                        "✅ V100 Safe" if MAX_NEW_TOKENS <= 4000 else "⚠️ High for V100")
    config_table.add_row("Model Parameters", f"{model_params:,.0f}", "✅ Loaded")
    config_table.add_row("Memory Optimization", "Enabled", "✅ low_cpu_mem_usage")
    
    console.print(config_table)
    
    # Model functionality test
    rprint("[dim]Running model compatibility test...[/dim]")
    try:
        # Test chat template processing
        test_messages = [
            {"role": "user", "content": [
                {"type": "text", "text": "Test"}
            ]}
        ]
        test_input_text = processor.apply_chat_template(test_messages, add_generation_prompt=True)
        
        if len(test_input_text) < 10:
            raise RuntimeError("Chat template processing failed")
        
        rprint("[green]✅ Model compatibility test passed[/green]")
        
    except Exception as e:
        rprint(f"[red]❌ Model compatibility test failed: {e}[/red]")
        raise RuntimeError(f"Model validation failed: {e}")

except Exception as e:
    rprint(f"[bold red]❌ CRITICAL ERROR: Model loading failed[/bold red]")
    rprint(f"[red]💥 Error: {e}[/red]")
    rprint("[yellow]💡 Check model path, GPU memory, and V100 compatibility requirements[/yellow]")
    raise  # Re-raise to prevent notebook execution with invalid model

rprint("[bold green]🎉 Model loading and validation complete![/bold green]")

🚀 Loading Llama Vision model with V100 production optimizations...

🔧 Configuring 8-bit quantization with BitsAndBytesConfig

✅ Modern BitsAndBytesConfig configured

Loading Llama-3.2-Vision model...

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

...

✅ Good GPU memory usage: 3.2%

                                     🔧 Production Model Configuration                                      
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting             ┃ Value                                                       ┃ Status               ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ Model Path          │ /home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct │ ✅ Valid             │
│ Device Placement    │ cuda:0                                                      │ ✅ Loaded            │
│ Quantization        │ 8-bit BitsAndBytesConfig                                    │ ✅ Modern API        │
│ Max New Tokens      │ 4000                                                        │ ✅ V100 Safe         │
│ Model Parameters    │ 10,670,220,835                                              │ ✅ Loaded            │
│ Memory Optimization │ Enabled                                                     │ ✅ low_cpu_mem_usage │
└─────────────────────┴─────────────────────────────────────────────────────────────┴──────────────────────┘

Running model compatibility test...

✅ Model compatibility test passed

🎉 Model loading and validation complete!

# Prompt Engineering

Define the optimized prompt for bank statement transaction extraction with clear visual guidance.

In [5]:
# ============================================================================= 
# MINIMAL PROMPT - SEPARATE TABLES PER DATE (BEST WORKING VERSION)
# =============================================================================

MINIMAL_PROMPT = """Extract the transaction table from this bank statement.

# CRITICAL: Create a SEPARATE markdown table for EACH DISTINCT DATE.

IMPORTANT: Look at the ACTUAL COLUMNS in the bank statement image:
- Column 1: Date
- Column 2: Transaction (description)
- Column 3: Debit (amount only appears here if money LEAVES account)
- Column 4: Credit (amount only appears here if money ENTERS account)
- Column 5: Balance (running balance after transaction)

READ THE VISUAL TABLE CAREFULLY:
- If you see an amount in the DEBIT column → put it in Debit, Credit = NOT_FOUND
- If you see an amount in the CREDIT column → put it in Credit, Debit = NOT_FOUND
- ALWAYS copy the balance value exactly as shown

# For each date:
# 1. Create a new markdown table
# 2. If multiple transactions occur on that date, each gets its own row
# 3. Copy the amounts EXACTLY as they appear in the original columns

Example:
### Date: [date]
| Date | Description | Debit | Credit | Balance |
|------|-------------|-------|--------|---------|
| [date] | [description] | [if amount in debit column] | [if amount in credit column] | [balance] |

Use NOT_FOUND only for empty debit/credit cells.
DO NOT classify based on transaction description - use the ACTUAL column position from the image.
"""

print("📋 MINIMAL PROMPT:")
print("="*60)
print(MINIMAL_PROMPT)
print("="*60)

📋 MINIMAL PROMPT:
Extract the transaction table from this bank statement.

# CRITICAL: Create a SEPARATE markdown table for EACH DISTINCT DATE.

IMPORTANT: Look at the ACTUAL COLUMNS in the bank statement image:
- Column 1: Date
- Column 2: Transaction (description)
- Column 3: Debit (amount only appears here if money LEAVES account)
- Column 4: Credit (amount only appears here if money ENTERS account)
- Column 5: Balance (running balance after transaction)

READ THE VISUAL TABLE CAREFULLY:
- If you see an amount in the DEBIT column → put it in Debit, Credit = NOT_FOUND
- If you see an amount in the CREDIT column → put it in Credit, Debit = NOT_FOUND
- ALWAYS copy the balance value exactly as shown

# For each date:
# 1. Create a new markdown table
# 2. If multiple transactions occur on that date, each gets its own row
# 3. Copy the amounts EXACTLY as they appear in the original columns

Example:
### Date: [date]
| Date | Description | Debit | Credit | Balance |
|------|-------------|-

In [6]:
# =============================================================================
# LLAMAVISIONTABLEEXTRACTOR CLASS - UNIFIED EXTRACTION IMPLEMENTATION
# =============================================================================

from datetime import datetime
from typing import Dict, List, Any, Optional

class LlamaVisionTableExtractor:
    """
    Production-ready class for extracting structured data from bank statements
    using Llama 3.2 Vision model.
    
    Features:
    - Comprehensive error handling and validation
    - GPU memory monitoring and V100 optimization
    - Ground truth integration for testing
    - Flexible initialization (standalone or notebook integration)
    """
    
    def __init__(self, model_path: str = None, processor=None, model=None):
        """
        Initialize with either a model path OR existing processor/model instances.
        
        Args:
            model_path: Path to load new model from
            processor: Existing processor instance 
            model: Existing model instance
        """
        if processor is not None and model is not None:
            # Use existing instances from notebook
            self.processor = processor
            self.model = model
            rprint("[green]✅ Using existing model and processor instances[/green]")
        elif model_path:
            # Load new instances (for standalone usage)
            rprint(f"[yellow]Loading new model from: {model_path}[/yellow]")
            self.processor = AutoProcessor.from_pretrained(model_path)
            self.model = MllamaForConditionalGeneration.from_pretrained(
                model_path,
                torch_dtype=torch.bfloat16,
                device_map="auto"
            )
            rprint("[green]✅ Standalone model and processor loaded[/green]")
        else:
            raise ValueError("Must provide either model_path OR (processor, model) instances")
    
    def extract_table(self, 
                     image_path: str, 
                     prompt_template: str = None,
                     max_new_tokens: int = None,
                     temperature: float = 0.1) -> Dict[str, Any]:
        """
        Extract structured data from table image using Llama 3.2 Vision.
        
        Args:
            image_path: Path to bank statement image
            prompt_template: Custom prompt (defaults to MINIMAL_PROMPT)
            max_new_tokens: Maximum tokens to generate (uses global MAX_NEW_TOKENS if None)
            temperature: Generation temperature (lower = more deterministic)
            
        Returns:
            Dictionary with extraction results
            
        Raises:
            FileNotFoundError: If image file doesn't exist
            RuntimeError: If model inference fails
            ValueError: If inputs are invalid
        """
        try:
            # Use global configuration for token limits if not specified
            if max_new_tokens is None:
                max_new_tokens = MAX_NEW_TOKENS
                rprint(f"[dim]Using global MAX_NEW_TOKENS: {max_new_tokens}[/dim]")
            
            # Validate inputs
            if not image_path:
                raise ValueError("Image path cannot be empty")
            
            image_path = Path(image_path)
            if not image_path.exists():
                raise FileNotFoundError(f"Image file not found: {image_path}")
            
            # Load and validate image
            rprint(f"[dim]Loading image: {image_path.name}...[/dim]")
            try:
                image = Image.open(image_path)
                # Validate image format
                if image.format not in ['PNG', 'JPEG', 'JPG', None]:
                    rprint(f"[yellow]⚠️ Warning: Unusual image format {image.format}[/yellow]")
            except Exception as e:
                raise ValueError(f"Failed to load image {image_path}: {e}")
            
            # Use provided prompt or default to MINIMAL_PROMPT
            prompt = prompt_template if prompt_template else MINIMAL_PROMPT
            
            # Prepare inputs with error handling
            try:
                messages = [
                    {"role": "user", "content": [
                        {"type": "image"},
                        {"type": "text", "text": prompt}
                    ]}
                ]
                
                # Apply chat template
                input_text = self.processor.apply_chat_template(messages, add_generation_prompt=True)
                
                # Process inputs
                inputs = self.processor(
                    image,
                    input_text,
                    add_special_tokens=False,
                    return_tensors="pt"
                ).to(self.model.device)
                
            except Exception as e:
                raise RuntimeError(f"Failed to process inputs: {e}")
            
            # Generate response with comprehensive error handling
            rprint(f"[yellow]🔍 Extracting data from {image_path.name} (max_tokens: {max_new_tokens})...[/yellow]")
            try:
                with torch.no_grad():
                    # Check GPU memory before inference
                    if torch.cuda.is_available():
                        memory_before = torch.cuda.memory_allocated() / 1e9
                        if memory_before > 15:  # Warn if using >15GB (V100 constraint)
                            rprint(f"[yellow]⚠️ High GPU memory usage: {memory_before:.1f}GB[/yellow]")
                    
                    output = self.model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        temperature=temperature,
                        top_p=0.95
                    )
                    
            except torch.cuda.OutOfMemoryError as e:
                raise RuntimeError(f"GPU out of memory. Try reducing max_new_tokens from {max_new_tokens}: {e}")
            except Exception as e:
                raise RuntimeError(f"Model inference failed: {e}")
            
            # Decode response with error handling
            try:
                response = self.processor.decode(output[0], skip_special_tokens=True)
                
                # Extract assistant response (clean up chat template artifacts)
                if "assistant" in response:
                    response = response.split("assistant")[-1].strip()
                
                # Validate response length
                if len(response.strip()) < 10:
                    rprint("[yellow]⚠️ Warning: Very short model response - possible inference issue[/yellow]")
                
            except Exception as e:
                raise RuntimeError(f"Failed to decode model response: {e}")
            
            # Parse and return structured response
            return self._parse_response(response, str(image_path), prompt, temperature, max_new_tokens)
            
        except Exception as e:
            # Log error and re-raise with context
            rprint(f"[red]❌ Extraction failed for {image_path}: {e}[/red]")
            raise
    
    def _parse_response(self, response: str, image_path: str, prompt: str, temperature: float, max_new_tokens: int) -> Dict[str, Any]:
        """Parse model response into structured format."""
        parsed_data = self._extract_table_data(response)
        return {
            "raw_response": response,
            "image_path": image_path,
            "extraction_time": datetime.now().isoformat(),
            "parsed_data": parsed_data,
            "transaction_count": len(parsed_data),
            "prompt_used": prompt[:100] + "..." if len(prompt) > 100 else prompt,
            "temperature": temperature,
            "max_new_tokens": max_new_tokens,
            "response_length": len(response)
        }
    
    def _extract_table_data(self, response: str) -> List[Dict]:
        """Extract table rows from markdown response."""
        lines = response.split('\n')
        table_rows = []
        
        for line in lines:
            if '|' in line and not line.strip().startswith('|---') and 'Date' not in line:
                # Parse table row
                cells = [cell.strip() for cell in line.split('|')[1:-1]]
                if len(cells) >= 4:  # Ensure minimum columns
                    row_data = {
                        'date': cells[0] if cells[0] != 'NOT_FOUND' else None,
                        'description': cells[1] if cells[1] != 'NOT_FOUND' else None,
                        'debit': cells[2] if cells[2] != 'NOT_FOUND' else None,
                        'credit': cells[3] if cells[3] != 'NOT_FOUND' else None,
                        'balance': cells[4] if len(cells) > 4 and cells[4] != 'NOT_FOUND' else None
                    }
                    # Only add if at least one field has data
                    if any(value is not None for value in row_data.values()):
                        table_rows.append(row_data)
        
        return table_rows
    
    def batch_extract(self, image_paths: List[str], prompt_template: str = None, max_new_tokens: int = None) -> Dict[str, Any]:
        """
        Extract from multiple images and return consolidated results.
        
        Args:
            image_paths: List of image file paths to process
            prompt_template: Extraction prompt to use (defaults to MINIMAL_PROMPT)
            max_new_tokens: Token limit per extraction (uses global MAX_NEW_TOKENS if None)
            
        Returns:
            Dictionary with individual results and summary statistics
        """
        # Use global configuration if not specified
        if max_new_tokens is None:
            max_new_tokens = MAX_NEW_TOKENS
        
        results = {}
        total_transactions = 0
        successful_extractions = 0
        start_time = datetime.now()
        
        rprint(f"[bold cyan]🔄 Processing {len(image_paths)} images (max_tokens: {max_new_tokens})...[/bold cyan]")
        
        for i, image_path in enumerate(image_paths, 1):
            try:
                rprint(f"[dim]Processing {i}/{len(image_paths)}: {Path(image_path).name}[/dim]")
                
                result = self.extract_table(image_path, prompt_template, max_new_tokens)
                results[image_path] = result
                total_transactions += result['transaction_count']
                successful_extractions += 1
                
                rprint(f"[green]✅ {Path(image_path).name}: {result['transaction_count']} transactions[/green]")
                
            except Exception as e:
                rprint(f"[red]❌ {Path(image_path).name}: Error - {e}[/red]")
                results[image_path] = {"error": str(e), "success": False}
        
        end_time = datetime.now()
        processing_time = (end_time - start_time).total_seconds()
        
        return {
            "individual_results": results,
            "summary": {
                "total_images": len(image_paths),
                "successful_extractions": successful_extractions,
                "total_transactions": total_transactions,
                "processing_time": processing_time,
                "average_time_per_image": processing_time / len(image_paths) if image_paths else 0,
                "success_rate": (successful_extractions / len(image_paths) * 100) if image_paths else 0,
                "max_new_tokens_used": max_new_tokens
            }
        }
    
    def test_extraction(self, image_path: str, prompt: str = None, max_new_tokens: int = None) -> Dict[str, Any]:
        """
        Run extraction test with comprehensive validation and metrics.
        
        Args:
            image_path: Path to test image
            prompt: Optional custom prompt
            max_new_tokens: Token limit (uses global MAX_NEW_TOKENS if None)
            
        Returns:
            Dictionary with test results and metrics
        """
        # Use global configuration if not specified
        if max_new_tokens is None:
            max_new_tokens = MAX_NEW_TOKENS
            
        if not image_path:
            rprint("[red]❌ No image path provided for testing[/red]")
            return {"error": "No image path"}
        
        try:
            rprint("[bold green]🧪 RUNNING EXTRACTION TEST[/bold green]")
            rprint(f"[cyan]📄 Image: {image_path}[/cyan]")
            rprint(f"[blue]🔧 Max tokens: {max_new_tokens}[/blue]")
            console.rule("[bold blue]Extraction Test[/bold blue]")
            
            # Run extraction
            start_time = datetime.now()
            result = self.extract_table(image_path, prompt_template=prompt or MINIMAL_PROMPT, max_new_tokens=max_new_tokens)
            end_time = datetime.now()
            processing_time = (end_time - start_time).total_seconds()
            
            # Display results with Rich formatting
            result_panel = Panel(
                result["raw_response"],
                title="[bold yellow]📊 EXTRACTION OUTPUT[/bold yellow]",
                border_style="yellow",
                expand=False
            )
            console.print(result_panel)
            
            # Get expected count from ground truth
            expected_count = get_expected_transaction_count(image_path)
            
            # Create comprehensive analysis table
            analysis_table = Table(title="📈 Extraction Analysis", border_style="green")
            analysis_table.add_column("Metric", style="cyan", no_wrap=True)
            analysis_table.add_column("Value", style="magenta")
            analysis_table.add_column("Status", justify="center")
            
            # Transaction count analysis
            extracted_count = result['transaction_count']
            count_status = "✅ Perfect Match" if extracted_count == expected_count else f"⚠️ Expected {expected_count}"
            analysis_table.add_row("Transactions Extracted", str(extracted_count), count_status)
            analysis_table.add_row("Ground Truth Count", str(expected_count), "📋 From CSV")
            analysis_table.add_row("Processing Time", f"{processing_time:.2f}s", "⏱️ Performance")
            analysis_table.add_row("Max Tokens Used", str(max_new_tokens), "🔧 Configuration")
            
            # Quality checks
            response_length = len(result["raw_response"])
            length_status = "✅ Good" if 100 < response_length < 5000 else "⚠️ Check Quality"
            analysis_table.add_row("Response Length", f"{response_length} chars", length_status)
            
            # Hallucination check
            hallucination_patterns = ["IGA", "1102.37"]
            hallucination_detected = any(
                pattern in result["raw_response"] and result["raw_response"].count(pattern) > 3 
                for pattern in hallucination_patterns
            )
            hallucination_status = "❌ Possible Hallucination" if hallucination_detected else "✅ Clean"
            analysis_table.add_row("Hallucination Check", "Pattern Detection", hallucination_status)
            
            console.print(analysis_table)
            
            # Add performance summary
            rprint(f"[green]✅ Test completed in {processing_time:.2f}s[/green]")
            
            return {
                "success": True,
                "extracted_count": extracted_count,
                "expected_count": expected_count,
                "accuracy": extracted_count == expected_count,
                "processing_time": processing_time,
                "response_length": response_length,
                "hallucination_detected": hallucination_detected,
                "max_new_tokens_used": max_new_tokens,
                "raw_result": result
            }
            
        except Exception as e:
            rprint(f"[red]❌ Test failed: {e}[/red]")
            return {"error": str(e), "success": False}

# =============================================================================
# GROUND TRUTH VALIDATION HELPER
# =============================================================================

def get_expected_transaction_count(image_path: str) -> int:
    """
    Get the expected number of transactions from ground truth CSV.
    
    Args:
        image_path: Path to the image file
        
    Returns:
        Expected transaction count (0 if not found or error)
    """
    try:
        if not Path(GROUND_TRUTH_CSV).exists():
            rprint(f"[yellow]⚠️ Ground truth file not found: {GROUND_TRUTH_CSV}[/yellow]")
            return 0
            
        # Read ground truth CSV
        ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV)
        
        # Extract just the filename from the path
        image_filename = Path(image_path).name
        
        # Find the row for this image
        image_row = ground_truth_df[ground_truth_df['image_file'] == image_filename]
        
        if image_row.empty:
            rprint(f"[yellow]⚠️ No ground truth found for {image_filename}[/yellow]")
            return 0
        
        # Count transactions - use TRANSACTION_DATES column
        transaction_dates = image_row.iloc[0]['TRANSACTION_DATES']
        
        if pd.isna(transaction_dates) or transaction_dates == 'NOT_FOUND':
            # Try LINE_ITEM_DESCRIPTIONS as fallback for receipts/invoices
            line_items = image_row.iloc[0]['LINE_ITEM_DESCRIPTIONS']
            if pd.isna(line_items) or line_items == 'NOT_FOUND':
                return 0
            return len(line_items.split(' | '))
        
        # Count the number of dates (transactions)
        return len(transaction_dates.split(' | '))
        
    except Exception as e:
        rprint(f"[red]❌ Error reading ground truth: {e}[/red]")
        return 0

In [7]:
# =============================================================================
# TEST THE LLAMAVISIONTABLEEXTRACTOR CLASS
# =============================================================================

rprint("[bold cyan]🧪 Testing LlamaVisionTableExtractor with current model...[/bold cyan]")

# Create extractor using existing model and processor from notebook
extractor = LlamaVisionTableExtractor(processor=processor, model=model)

console.rule("[bold blue]Class-based Extraction Test[/bold blue]")

# Test with the current image and minimal prompt
if STATEMENT_IMAGE_PATH:
    result = extractor.extract_table(STATEMENT_IMAGE_PATH, MINIMAL_PROMPT)
    
    # Display results
    rprint(f"[cyan]📄 Image processed:[/cyan] {result['image_path']}")
    rprint(f"[magenta]📊 Transactions found:[/magenta] {result['transaction_count']}")
    rprint(f"[green]⏰ Processing time:[/green] {result['extraction_time']}")
    
    # Show parsed data in a table
    if result['parsed_data']:
        parsed_table = Table(title="📋 Parsed Transaction Data", border_style="green")
        parsed_table.add_column("#", style="dim", width=3)
        parsed_table.add_column("Date", style="cyan")
        parsed_table.add_column("Description", style="white", max_width=30)
        parsed_table.add_column("Debit", style="red")
        parsed_table.add_column("Credit", style="green")
        parsed_table.add_column("Balance", style="yellow")
        
        for i, transaction in enumerate(result['parsed_data'], 1):
            parsed_table.add_row(
                str(i),
                transaction.get('date', ''),
                transaction.get('description', ''),
                transaction.get('debit', ''),
                transaction.get('credit', ''),
                transaction.get('balance', '')
            )
        
        console.print(parsed_table)
    else:
        rprint("[red]❌ No transactions parsed from response[/red]")
    
    # Show raw response
    raw_panel = Panel(
        result["raw_response"][:500] + "..." if len(result["raw_response"]) > 500 else result["raw_response"],
        title="[bold yellow]🔍 Raw Model Response (truncated)[/bold yellow]",
        border_style="yellow",
        expand=False
    )
    console.print(raw_panel)
    
    # Compare with ground truth
    expected_count = get_expected_transaction_count(STATEMENT_IMAGE_PATH)
    
    comparison_table = Table(title="📈 Class vs Function Comparison", border_style="blue")
    comparison_table.add_column("Method", style="cyan")
    comparison_table.add_column("Transactions", style="magenta")
    comparison_table.add_column("Match Ground Truth", style="green")
    
    comparison_table.add_row("Class Method", str(result['transaction_count']), 
                           "✅ Perfect" if result['transaction_count'] == expected_count else f"⚠️ Expected {expected_count}")
    
    console.print(comparison_table)

else:
    rprint("[red]❌ No image path available for testing[/red]")

console.rule("[bold blue]Next Steps[/bold blue]")
rprint("[green]💡 Class loaded successfully - ready for production use![/green]")
rprint("[cyan]💡 Try: extractor.batch_extract([image1, image2], MINIMAL_PROMPT)[/cyan]")

🧪 Testing LlamaVisionTableExtractor with current model...

✅ Using existing model and processor instances

─────────────────────────────────────────── Class-based Extraction Test ───────────────────────────────────────────

Using global MAX_NEW_TOKENS: 4000

Loading image: commbank_statement_001.png...

🔍 Extracting data from commbank_statement_001.png (max_tokens: 4000)...

❌ Extraction failed for evaluation_data/commbank_statement_001.png: Model inference failed: view size is not 
compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use 
.reshape(...) instead.

RuntimeError: Model inference failed: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.